# SocialJax Quickstart Tutorial (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cooperativex/SocialJax/blob/v2/tutorials/01_quickstart_colab.ipynb)

Welcome to SocialJax! This tutorial will get you up and running with multi-agent reinforcement learning in minutes.

## What is SocialJax?

SocialJax is a JAX-based framework for multi-agent reinforcement learning, featuring:
- **Sequential Social Dilemmas**: Environments that test cooperation vs. competition
- **Multiple Algorithms**: IPPO, MAPPO, VDN, and SVO
- **JAX Acceleration**: Fast training with GPU support
- **Flexible Configuration**: YAML configs and dataclass-based settings

## 0. Colab Setup

Run the cell below to install SocialJax and set up the environment. This will:
1. Clone the SocialJax repository
2. Install all dependencies including JAX with GPU support
3. Verify the installation

In [ ]:
# @title Colab Setup { display-mode: "form" }
# @markdown Run this cell to install SocialJax on Colab

import os
import sys

# Check if running on Colab
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    print("📥 Installing SocialJax on Google Colab...\n")
    
    # Clone the repository
    print("Step 1: Cloning repository...")
    !git clone -b v2 https://github.com/cooperativex/SocialJax.git
    %cd SocialJax
    
    # Install NumPy 1.x (required for JAX 0.4.x compatibility)
    # NumPy 2.0+ has breaking changes with copy=False parameter
    print("\nStep 2: Installing NumPy 1.26.x (compatible version)...")
    !pip install -q "numpy<2.0"
    
    # Install dependencies
    print("\nStep 3: Installing other dependencies...")
    !pip install -q -r requirements.txt
    
    # Install JAX 0.4.x (required for compatibility with SocialJax)
    # Note: JAX 0.6+ removed jax.tree_map which SocialJax uses
    print("\nStep 4: Installing JAX 0.4.30 with GPU support...")
    !pip install -q jax[cuda11_pip]==0.4.30 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    
    # Set PYTHONPATH
    sys.path.insert(0, '/content/SocialJax/socialjax')
    
    print("\n✅ SocialJax installed successfully!")
else:
    print("Not running on Colab. Make sure SocialJax is installed locally.")
    sys.path.insert(0, './socialjax')

In [ ]:
# @title Verify Installation { display-mode: "form" }

print("=" * 50)
print("VERIFICATION")
print("=" * 50)

# Check JAX
import jax
print(f"\n🔎 JAX version: {jax.__version__}")
print(f"    Devices: {jax.devices()}")

# Check for GPU/TPU (handle case when not available)
try:
    gpu_devices = jax.devices('gpu')
    if len(gpu_devices) > 0:
        print(f"    🚀 GPU acceleration enabled!")
except RuntimeError:
    pass

try:
    tpu_devices = jax.devices('tpu')
    if len(tpu_devices) > 0:
        print(f"    🚀 TPU acceleration enabled!")
except RuntimeError:
    pass

# If no GPU/TPU, show CPU message
if not any(d.platform in ('gpu', 'tpu') for d in jax.devices()):
    print(f"    ⚠️ Running on CPU. Training will be slower.")
    print(f"    💡 Tip: Enable GPU via Runtime > Change runtime type > GPU")

# Check SocialJax
import socialjax
from socialjax import make, registered_envs
print(f"\n🔎 SocialJax version: {socialjax.__version__}")

# Check available environments
print(f"\n🔎 Available environments ({len(registered_envs)}):")
for env_name in sorted(registered_envs):
    print(f"    - {env_name}")

# Quick environment test
print(f"\n🔎 Testing environment creation...")
try:
    test_env = make('coin_game', num_agents=2)
    # Note: observation_space() returns (Box, shape) tuple
    obs_space, obs_shape = test_env.observation_space()
    action_space = test_env.action_space()
    print(f"    ✅ Successfully created 'coin_game' environment!")
    print(f"    - Number of agents: {test_env.num_agents}")
    print(f"    - Observation shape: {obs_shape}")
    print(f"    - Number of actions: {action_space.n}")
except Exception as e:
    print(f"    ❌ Failed to create environment: {e}")

print("\n" + "=" * 50)
print("🎉 All systems ready! Let's start learning!")
print("=" * 50)

---

## 1. Import SocialJax

Now that everything is installed, let's import the modules we'll need.

In [ ]:
# Import the main module
import socialjax
from socialjax import make, registered_envs
import jax
import jax.numpy as jnp

print("Imports successful!")

## 2. Create an Environment

SocialJax provides several multi-agent environments. Let's create the **Coin Game** environment, a classic sequential social dilemma where agents must balance collecting coins vs. cooperating.

In [ ]:
# Create the Coin Game environment with 5 agents
env = make('coin_game', num_agents=5)

# Get observation and action spaces
# Note: observation_space() returns (Box, shape) tuple
obs_space, obs_shape = env.observation_space()
action_space = env.action_space()

print(f"Environment: Coin Game")
print(f"Number of agents: {env.num_agents}")
print(f"Observation shape: {obs_shape}")
print(f"Number of actions: {action_space.n}")

## 3. Explore the Environment

Let's reset the environment and take a few random steps to understand how it works.

In [ ]:
# Reset the environment
key = jax.random.PRNGKey(0)
obs, state = env.reset(key)

print(f"Initial observation shape: {obs.shape}")
print(f"State type: {type(state).__name__}")

# Take a random step
# Note: step() requires (key, state, actions)
key, action_key = jax.random.split(key)
actions = jax.random.randint(action_key, (env.num_agents,), 0, action_space.n)

# Step the environment - pass key as first argument
obs, state, rewards, dones, infos = env.step(key, state, actions)

print(f"\nAfter one step:")
print(f"Rewards: {rewards}")
print(f"Dones: {dones}")

## 4. Run Multiple Steps

Let's run a few more steps to see how the environment evolves.

In [ ]:
# Run 10 random steps
print("Running 10 random steps...\n")

total_reward = jnp.zeros(env.num_agents)
for step in range(10):
    key, action_key = jax.random.split(key)
    actions = jax.random.randint(action_key, (env.num_agents,), 0, action_space.n)
    # Note: step() requires key as first argument
    obs, state, rewards, dones, infos = env.step(key, state, actions)
    total_reward = total_reward + rewards
    
    if step % 3 == 0:
        print(f"Step {step}: rewards = {rewards}")

print(f"\nTotal rewards after 10 steps: {total_reward}")
print(f"Average reward per agent: {total_reward / 10}")

## 5. Create an Algorithm

SocialJax provides several multi-agent RL algorithms. Let's create an **IPPO** (Independent PPO) agent.

In [ ]:
from socialjax.algorithms import get_algorithm, list_algorithms

# List available algorithms
print("Available algorithms:")
for algo in list_algorithms():
    print(f"  - {algo}")

# Get the IPPO algorithm class
IPPOAlgorithm = get_algorithm('ippo')
print(f"\nUsing: IPPOAlgorithm")

## 6. Create a Trainer

The Trainer class provides a high-level interface for training algorithms.

In [ ]:
from socialjax.training import Trainer

# Create a trainer with IPPO on Coin Game
trainer = Trainer(
    algorithm='ippo',
    env='coin_game',
    num_agents=5,
    config={
        'total_timesteps': 10000,  # Short run for demo
        'num_envs': 4,
        'num_steps': 16,
        'learning_rate': 0.001,
        'gamma': 0.99,
    }
)

print("Trainer created successfully!")
print(f"Algorithm: {trainer.algorithm_name}")
print(f"Environment: {trainer.env_name}")

## 7. Train the Agent

Now let's train our IPPO agent on the Coin Game environment.

In [ ]:
# Train for a short run (this may take a minute)
print("Training IPPO on Coin Game...")
print("(This is a short demo run - increase total_timesteps for real training)\n")

# Note: For a quick demo, we'll train with minimal steps
# Real training typically uses 1M+ steps
metrics = trainer.train(total_timesteps=5000)

print("\nTraining complete!")
print(f"Total timesteps: {metrics.get('total_timesteps', 'N/A')}")
if 'mean_episode_return' in metrics:
    print(f"Mean episode return: {metrics['mean_episode_return']:.2f}")

## 8. Evaluate the Trained Agent

After training, let's evaluate our agent's performance.

In [ ]:
# Evaluate the trained agent
print("Evaluating trained agent...")

eval_metrics = trainer.evaluate(n_episodes=10)

print(f"\nEvaluation Results:")
print(f"  Mean return: {eval_metrics.mean_return:.2f}")
print(f"  Std return: {eval_metrics.std_return:.2f}")
if hasattr(eval_metrics, 'cooperation_rate'):
    print(f"  Cooperation rate: {eval_metrics.cooperation_rate:.2%}")

## 9. Save and Load Checkpoints

You can save and load model checkpoints for later use.

In [ ]:
import os

# Save checkpoint
checkpoint_path = "checkpoints/quickstart_ippo"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

trainer.save(checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")

# Load checkpoint (demonstration)
new_trainer = Trainer(algorithm='ippo', env='coin_game', num_agents=5)
new_trainer.load(checkpoint_path)
print(f"Checkpoint loaded successfully!")

## 10. Visualize Agent Behavior

You can generate GIFs to visualize your trained agent's behavior.

In [ ]:
from socialjax.evaluation import save_gif

# Run evaluation with frame capture
# Note: This requires a trained agent
print("To generate visualizations, use the CLI tool:")
print("python scripts/visualize.py --checkpoint checkpoints/quickstart_ippo --env coin_game --output output.gif")

## 11. Other Environments

SocialJax provides several environments for studying social dilemmas:

In [ ]:
# List all environments with descriptions
environments = {
    'coin_game': 'Collect coins while avoiding conflict',
    'clean_up': 'Clean pollution to harvest apples',
    'harvest_common_open': 'Harvest apples from common resource',
    'coop_mining': 'Cooperative mining for resources',
    'territory_open': 'Territory control and resource gathering',
    'pd_arena': 'Prisoner\'s Dilemma in spatial arena',
    'mushrooms': 'Harvest mushrooms sustainably',
    'gift': 'Gift-giving and social learning',
}

print("Available Environments:\n")
for name, desc in environments.items():
    print(f"  {name}: {desc}")

## 12. Command-Line Training

For production training, use the CLI scripts:

In [ ]:
# CLI training example
print("Training with CLI:")
print("""\npython scripts/train.py \\
    --algorithm ippo \\
    --env coin_game \\
    --timesteps 1000000 \\
    --num-envs 8 \\
    --wandb-project socialjax-experiments
""")

print("\nEvaluation with CLI:")
print("""\npython scripts/evaluate.py \\
    --checkpoint checkpoints/ippo_final \\
    --env coin_game \\
    --episodes 50 \\
    --render \\
    --output evaluation.gif
""")

---

## Summary

In this quickstart, you learned:

1. How to **install SocialJax** in Google Colab with GPU support
2. How to **create environments** with `socialjax.make()`
3. How to **create algorithms** using the registry
4. How to **train agents** with the Trainer class
5. How to **evaluate and save** trained models
6. How to use the **CLI tools** for production training

## Next Steps

- **Tutorial 2**: Creating custom algorithms
- **Tutorial 3**: Creating custom network architectures
- **Tutorial 4**: Using callbacks for logging and monitoring
- **Tutorial 5**: Advanced configuration options

## Resources

- [API Reference](../docs/api_reference.md)
- [Migration Guide](../docs/migration_guide.md)
- [GitHub Repository](https://github.com/cooperativex/SocialJax)